In [1]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers import processors

tokenizer = Tokenizer(BPE())

trainer = BpeTrainer(
    vocab_size=10**4,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
    initial_alphabet=ByteLevel.alphabet()
)

tokenizer.pre_tokenizer = ByteLevel(
    add_prefix_space=False,
    use_regex=True
)

In [2]:
# files = [f"./cying/datasets/wikitext-103-raw/wiki.{split}.raw" for split in ["test", "train", "valid"]]
files = [f"./cying/datasets/translation2019zh/translation2019zh_valid.json"]

tokenizer.train(files, trainer)
tokenizer.post_processor = processors.ByteLevel(trim_offsets=True)

In [3]:
tokenizer.save("./cying/datasets/tokenizer-wiki.json")

In [5]:
tokenizer = Tokenizer.from_file("./cying/datasets/tokenizer-wiki.json")

In [6]:
def bytes_to_unicode():
    """
    Returns list of utf-8 byte and a mapping to unicode strings. We specifically avoids mapping to whitespace/control
    characters the bpe code barfs on.

    The reversible bpe codes work on unicode strings. This means you need a large # of unicode characters in your vocab
    if you want to avoid UNKs. When you're at something like a 10B token dataset you end up needing around 5K for
    decent coverage. This is a significant percentage of your normal, say, 32K bpe vocab. To avoid that, we want lookup
    tables between utf-8 bytes and unicode strings.
    """
    bs = (
        list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    )
    cs = bs[:]
    n = 0
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8 + n)
            n += 1
    cs = [chr(n) for n in cs]
    return dict(zip(bs, cs))
def bytes_to_unicode():
    # 原始保留的字节范围
    bs = (
        list(range(ord("!"), ord("~") + 1)) +          # ASCII可打印字符（33-126）
        list(range(ord("¡"), ord("¬") + 1)) +          # 西班牙语特殊字符（161-172）
        list(range(ord("®"), ord("ÿ") + 1))            # 其他扩展字符（174-255）
    )
    
    cs = bs.copy()  # 初始字符列表
    n = 0
    
    # 遍历所有可能的字节（0-255）
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8 + n)  # 超出原始范围的字节映射到更高Unicode码位
            n += 1
    
    # 将码位转换为Unicode字符
    cs = [chr(code) for code in cs]
    
    return dict(zip(bs, cs))

def get_reverse_mapping(forward_map):
    """
    根据正向映射生成反向映射
    返回字典：{unicode_char: byte_value}
    """
    return {v: k for k, v in forward_map.items()}

# 生成映射表
forward_map = bytes_to_unicode()
reverse_map = get_reverse_mapping(forward_map)

In [7]:
def unicode_str_to_bytes(unicode_str):
    return bytes([reverse_map[c] for c in unicode_str])
unicode_str = tokenizer.decode(tokenizer.encode('你好，我是王恒。你是谁？中央细胞以后的发育主要是极核的发育和极核周围胞质的变化。').ids).replace(' ','').replace('Ġ',' ')
# 将转换后的Unicode字符串还原为原始文本
recovered_bytes = unicode_str_to_bytes(unicode_str)
recovered_text = recovered_bytes.decode('utf-8')
print(f"还原后的文本：{recovered_text}")
# 输出：还原后的文本：你好

还原后的文本：你好，我是王恒。你是谁？中央细胞以后的发育主要是极核的发育和极核周围胞质的变化。
